# 01 — Download raw data

Sources (all public):

| Data | Source | Coverage |
|---|---|---|
| DB1B Market, DB1B Ticket (10% ticket sample) | BTS TranStats `PREZIP/` | 2022Q1–2025Q2 (latest posted) |
| T-100 Domestic Segment (U.S. carriers; foreign carriers cannot fly U.S. domestic segments) | BTS TranStats download form (table `FIM`) | 2022–2025, all months |
| Aviation Support Tables: Master Coordinate | BTS TranStats download form (table `FLL`) | current |
| Metro/micro area population estimates, Vintage 2025 | Census Bureau `cbsa-est2025-alldata.csv` | 2020–2025 |
| CBSA cartographic boundaries 2024 (1:500k) | Census Bureau `cb_2024_us_cbsa_500k.zip` | July 2023 delineation |
| CPI-U, all items, seasonally adjusted (`CPIAUCSL`) | FRED | monthly |

Files already present are not downloaded again. Raw files are kept out of git (see `.gitignore`).

In [1]:
%run 00_setup.ipynb

In [2]:
import re, zipfile, requests

S = requests.Session()
S.headers.update({"User-Agent": "Mozilla/5.0 (research download; airline merger project)"})
TRANSTATS = "https://www.transtats.bts.gov"

def fetch(url, dest, min_bytes=1000, **kw):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > min_bytes:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(4):
        try:
            with S.get(url, stream=True, timeout=600, **kw) as r:
                r.raise_for_status()
                tmp = dest.with_suffix(dest.suffix + ".part")
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            tmp.rename(dest)
            return dest
        except Exception as e:
            print("retry", attempt, url, e)
            time.sleep(2 ** (attempt + 1))
    raise RuntimeError(f"download failed: {url}")

## DB1B Market and Ticket (pre-zipped quarterly files)

In [3]:
db1b_dir = RAW / "db1b"
for table in ("Market", "Ticket"):
    for (y, q) in QUARTERS:
        name = f"Origin_and_Destination_Survey_DB1B{table}_{y}_{q}.zip"
        p = fetch(f"https://transtats.bts.gov/PREZIP/{name}", db1b_dir / name)
        print(f"{name}: {p.stat().st_size/1e6:,.0f} MB")

Origin_and_Destination_Survey_DB1BMarket_2022_1.zip: 82 MB
Origin_and_Destination_Survey_DB1BMarket_2022_2.zip: 99 MB
Origin_and_Destination_Survey_DB1BMarket_2022_3.zip: 94 MB
Origin_and_Destination_Survey_DB1BMarket_2022_4.zip: 95 MB
Origin_and_Destination_Survey_DB1BMarket_2023_1.zip: 89 MB
Origin_and_Destination_Survey_DB1BMarket_2023_2.zip: 100 MB
Origin_and_Destination_Survey_DB1BMarket_2023_3.zip: 98 MB
Origin_and_Destination_Survey_DB1BMarket_2023_4.zip: 100 MB
Origin_and_Destination_Survey_DB1BMarket_2024_1.zip: 95 MB
Origin_and_Destination_Survey_DB1BMarket_2024_2.zip: 110 MB
Origin_and_Destination_Survey_DB1BMarket_2024_3.zip: 108 MB
Origin_and_Destination_Survey_DB1BMarket_2024_4.zip: 109 MB
Origin_and_Destination_Survey_DB1BMarket_2025_1.zip: 95 MB
Origin_and_Destination_Survey_DB1BMarket_2025_2.zip: 110 MB
Origin_and_Destination_Survey_DB1BTicket_2022_1.zip: 76 MB
Origin_and_Destination_Survey_DB1BTicket_2022_2.zip: 93 MB
Origin_and_Destination_Survey_DB1BTicket_2022_3.zi

## TranStats download form (T-100 segment and Master Coordinate)

The form is an ASP.NET page: we read its hidden state fields, tick the variables we need and post it back.
The server answers with a zipped CSV.

In [4]:
def transtats_form(table_code, fields, dest, year=None, period=None):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 1000:
        return dest
    url = f"{TRANSTATS}/DL_SelectFields.aspx?gnoyr_VQ={table_code}&QO_fu146_anzr=b0-gvzr"
    page = S.get(url, timeout=120).text
    data = {k: v for k, v in re.findall(r'<input type="hidden" name="([^"]+)" id="[^"]*" value="([^"]*)"', page)}
    if "cboYear" in page:
        data.update({"cboGeography": "All", "cboYear": str(year or "All"), "cboPeriod": str(period or "All")})
    data.update({"chkDownloadZip": "on", "btnDownload": "Download"})
    data.update({f: "on" for f in fields})
    r = S.post(url, data=data, timeout=1800)
    r.raise_for_status()
    if not r.content[:2] == b"PK":
        raise RuntimeError(f"form did not return a zip for {table_code} {year}: {r.text[:300]}")
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(r.content)
    return dest

T100_FIELDS = ["DEPARTURES_SCHEDULED", "DEPARTURES_PERFORMED", "SEATS", "PASSENGERS", "DISTANCE",
               "UNIQUE_CARRIER", "CARRIER", "ORIGIN_AIRPORT_ID", "ORIGIN", "DEST_AIRPORT_ID", "DEST",
               "AIRCRAFT_TYPE", "AIRCRAFT_CONFIG", "YEAR", "QUARTER", "MONTH", "CLASS"]
for y in sorted({y for y, _ in QUARTERS}):
    p = transtats_form("FIM", T100_FIELDS, RAW / "t100" / f"T100_domestic_segment_{y}.zip", year=y, period="All")
    print(p.name, f"{p.stat().st_size/1e6:,.1f} MB")

COORD_FIELDS = ["AIRPORT_ID", "AIRPORT", "DISPLAY_AIRPORT_NAME", "DISPLAY_AIRPORT_CITY_NAME_FULL",
                "AIRPORT_STATE_CODE", "AIRPORT_COUNTRY_CODE_ISO", "LATITUDE", "LONGITUDE",
                "AIRPORT_IS_CLOSED", "AIRPORT_IS_LATEST", "CITY_MARKET_ID"]
p = transtats_form("FLL", COORD_FIELDS, RAW / "airports" / "master_coordinate.zip")
print(p.name, f"{p.stat().st_size/1e6:,.1f} MB")

T100_domestic_segment_2022.zip 4.7 MB
T100_domestic_segment_2023.zip 4.8 MB
T100_domestic_segment_2024.zip 4.9 MB
T100_domestic_segment_2025.zip 5.2 MB
master_coordinate.zip 0.3 MB


## Census population and CBSA boundaries; CPI

In [5]:
fetch("https://www2.census.gov/programs-surveys/popest/datasets/2020-2025/metro/totals/cbsa-est2025-alldata.csv",
      RAW / "census" / "cbsa-est2025-alldata.csv")
fetch("https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_cbsa_500k.zip",
      RAW / "census" / "cb_2024_us_cbsa_500k.zip")
# FRED refuses some browser-like user agents; a plain client string works.
fetch("https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL", RAW / "cpi" / "CPIAUCSL.csv", min_bytes=100,
      headers={"User-Agent": "curl/8.5.0"})

PosixPath('/home/user/BLP-/data/raw/cpi/CPIAUCSL.csv')

## Inventory

In [6]:
inv = pd.DataFrame([{"file": str(p.relative_to(ROOT)), "MB": round(p.stat().st_size / 1e6, 1)}
                    for p in sorted(RAW.rglob("*")) if p.is_file() and not p.name.endswith(".part")])
inv.to_csv(RES / "raw_inventory.csv", index=False)
print(f"{len(inv)} files, {inv.MB.sum()/1e3:,.2f} GB")
inv

36 files, 2.71 GB


,file,MB
0,data/raw/airports/master_coordinate.zip,0.3
1,data/raw/census/cb_2024_us_cbsa_500k.zip,4.7
2,data/raw/census/cbsa-est2025-alldata.csv,0.8
3,data/raw/cpi/CPIAUCSL.csv,0.0
4,data/raw/db1b/Origin_and_Destination_Survey_DB...,81.9
5,data/raw/db1b/Origin_and_Destination_Survey_DB...,99.1
6,data/raw/db1b/Origin_and_Destination_Survey_DB...,94.2
7,data/raw/db1b/Origin_and_Destination_Survey_DB...,95.2
8,data/raw/db1b/Origin_and_Destination_Survey_DB...,88.5
9,data/raw/db1b/Origin_and_Destination_Survey_DB...,100.2
